# Conformational control of HOMO–VBM alignment in carbazole 2D perovskites

Analysis of `cz-c_3`, `cz-c_4`, and `cz-c_5` using SATool structural descriptors and VASP-derived VBM/HOMO/CBM values.

**Input note:** headerless DFT reports use labels such as `c3_0025_C0`. The RSS structure ID is **25** (not the terminal conformer label `C0`). The reader below extracts the number between the first underscore and `_C<number>`.


In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.stats import spearmanr, linregress, bootstrap
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RepeatedKFold, cross_val_score
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('results')
OUTPUT_DIR = Path('analysis_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_WARD_CLUSTERS = 5
RANDOM_STATE = 42

# SATool has no explicit structure-ID column. Set 0 if its first row is RSS structure 0000; set 1 if it is 0001.
SATOOL_INDEX_BASE = 0

COMPOUND_PATHS = {
    'cz-c_3': {'satool': DATA_DIR / 'satool_props_c3.csv', 'dft': DATA_DIR / 'c_3-dft-eigenvalues.dat'},
    'cz-c_4': {'satool': DATA_DIR / 'satool_props_c4.csv', 'dft': DATA_DIR / 'c_4-dft-eigenvalues.dat'},
    'cz-c_5': {'satool': DATA_DIR / 'satool_props_c5.csv', 'dft': DATA_DIR / 'c_5-dft-eigenvalues.dat'},
}

DFT_ALIASES = {
    'structure_id': ['structure_id', 'structure', 'id', 'config_id', 'configuration_id', 'filename', 'file', 'name', 'index'],
    'vbm': ['vbm', 'e_vbm', 'vbm_ev', 'vbm_energy', 'valence_band_maximum'],
    'homo': ['homo', 'e_homo', 'homo_ev', 'homo_energy', 'organic_homo'],
    'cbm': ['cbm', 'e_cbm', 'cbm_ev', 'cbm_energy', 'conduction_band_minimum'],
}

SELECTION_RULES = {
    'trans_angle': {'keywords': ['trans', 'angle'], 'prefer': ['mean'], 'avoid': ['std', 'min', 'max', 'n']},
    'shared_mxm_angle': {'keywords': ['shared', 'halide', 'angle'], 'prefer': ['mean', 'in_plane'], 'avoid': ['std', 'min', 'max', 'n']},
    'chain_tail_dist': {'keywords': ['chain', 'tail'], 'prefer': ['mean', 'inorganic', 'plane', 'dist'], 'avoid': ['std', 'min', 'max', 'n']},
    'tilt_fragment': {'keywords': ['whole_fragment', 'inorganic_layer_plane'], 'prefer': ['mean'], 'avoid': ['std', 'min', 'max']},
    'tilt_core': {'keywords': ['ring_core', 'inorganic_layer_plane'], 'prefer': ['mean'], 'avoid': ['std', 'min', 'max']},
    'tilt_chain': {'keywords': ['chain_principal', 'inorganic_layer_plane'], 'prefer': ['mean'], 'avoid': ['std', 'min', 'max']},
    'hook_elevation': {'keywords': ['hook', 'elevation'], 'prefer': ['abs_mean', 'inorganic', 'plane'], 'avoid': ['std', 'min', 'max', 'n']},
}


## Functions
The parser accepts both headered tables and headerless DFT reports. It validates IDs before the structural/electronic merge.


In [ ]:
FLOAT_PATTERN = r'[-+]?(?:\d*\.\d+|\d+\.?\d*)(?:[Ee][-+]?\d+)?'

def normalise_name(value):
    return re.sub(r'[^a-z0-9]+', '_', str(value).lower()).strip('_')

def canonicalize_id(values):
    """Return the RSS ID from labels such as c3_0025_C0 -> 25.

    The terminal C0/C1/C2/C3 is a conformer suffix and must not be used
    as the merge key. Plain numeric IDs are also supported.
    """
    text = pd.Series(values, dtype='string').str.strip()
    # Main pattern: c3_0025_C0, c4_0142_C2, c5_1002_C3.
    rss_id = text.str.extract(r'^[A-Za-z]+\d*_(\d+)_C\d+$', expand=False)
    # Plain integer field / fallback for labels ending in an RSS integer.
    fallback_id = text.str.extract(r'(\d+)(?!.*\d)', expand=False)
    result = rss_id.fillna(fallback_id).fillna(text)
    return result.str.lstrip('0').replace('', '0').astype(str)

def find_column(columns, aliases, required=True):
    normalized = {normalise_name(c): c for c in columns}
    for alias in aliases:
        if normalise_name(alias) in normalized:
            return normalized[normalise_name(alias)]
    matches = list(dict.fromkeys(original for alias in aliases for norm, original in normalized.items() if normalise_name(alias) in norm))
    if len(matches) == 1:
        return matches[0]
    if required:
        raise KeyError(f'Could not identify a required column. Tried {aliases}. Available: {list(columns)}')
    return None

def score_satool_column(column, rule):
    name = normalise_name(column)
    if any(word not in name for word in rule['keywords']):
        return -np.inf
    return 10 * len(rule['keywords']) + 3 * sum(w in name for w in rule.get('prefer', [])) - 10 * sum(w in name for w in rule.get('avoid', []))

def infer_satool_columns(df):
    result = {}
    for canonical, rule in SELECTION_RULES.items():
        candidates = {col: score_satool_column(col, rule) for col in df.columns}
        candidates = {col: score for col, score in candidates.items() if np.isfinite(score)}
        result[canonical] = max(candidates, key=candidates.get) if candidates else None
        print(f'{canonical:20s} <- {result[canonical]}')
    return result

def load_satool(path):
    raw = pd.read_csv(path)
    raw.columns = [str(c).strip() for c in raw.columns]
    id_column = find_column(raw.columns, DFT_ALIASES['structure_id'], required=False)
    if id_column is None:
        print(f'WARNING: SATool structure identifier not found; assigning row IDs with base {SATOOL_INDEX_BASE}.')
        ids = pd.Series(np.arange(SATOOL_INDEX_BASE, SATOOL_INDEX_BASE + len(raw)), index=raw.index).astype(str)
    else:
        ids = canonicalize_id(raw[id_column])
    mapping = infer_satool_columns(raw)
    out = pd.DataFrame({'structure_id': ids})
    for canonical, source in mapping.items():
        if source is not None:
            out[canonical] = pd.to_numeric(raw[source], errors='coerce').to_numpy()
    return out.copy()

def noncomment_lines(path):
    return [line.strip() for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip() and not line.lstrip().startswith(('#', '!', ';'))]

def parse_headerless_dft(path):
    records = []
    for line_no, line in enumerate(noncomment_lines(path), start=1):
        tokens = [token for token in re.split(r'[\s,;|]+', line) if token]
        if not tokens:
            continue
        structure_label = tokens[0]
        numbers = [float(value) for value in re.findall(FLOAT_PATTERN, ' '.join(tokens[1:]))]
        if len(numbers) < 3:
            continue
        # Assumption from the supplied report format: final 3 numeric fields are VBM/HOMO/CBM.
        vbm, homo, cbm = numbers[-3:]
        records.append({'source_line': line_no, 'structure_label': structure_label, 'structure_id': canonicalize_id([structure_label]).iloc[0], 'vbm': vbm, 'homo': homo, 'cbm': cbm})
    if not records:
        raise ValueError('No DFT records could be parsed. Inspect the first raw lines and adapt parse_headerless_dft().')
    out = pd.DataFrame(records)
    if out['structure_id'].duplicated().any():
        duplicate = out.loc[out['structure_id'].duplicated(keep=False), ['source_line', 'structure_label', 'structure_id']]
        raise ValueError('Duplicate RSS IDs remain after parsing. This means multiple conformers share an RSS ID and require an explicit conformer key in SATool as well.\n' + duplicate.to_string(index=False))
    return out

def load_dft(path):
    try:
        candidate = pd.read_csv(path, sep=None, engine='python', comment='#')
        candidate.columns = [str(c).strip() for c in candidate.columns]
        selected = {key: find_column(candidate.columns, aliases, required=False) for key, aliases in DFT_ALIASES.items()}
        if all(selected.values()) and not any(str(c).startswith('Unnamed:') for c in candidate.columns):
            out = candidate[list(selected.values())].rename(columns={value: key for key, value in selected.items()}).copy()
            out['structure_id'] = canonicalize_id(out['structure_id'])
            for col in ['vbm', 'homo', 'cbm']:
                out[col] = pd.to_numeric(out[col], errors='raise')
            return out
    except Exception:
        pass
    print('INFO: Parsing headerless DFT report; final three numerical fields are assumed to be VBM, HOMO, CBM.')
    return parse_headerless_dft(path)

def add_electronic_metrics(df):
    out = df.copy()
    out['delta_homo_vbm'] = out['homo'] - out['vbm']
    out['abs_delta_homo_vbm'] = out['delta_homo_vbm'].abs()
    out['inorganic_gap'] = out['cbm'] - out['vbm']
    out['effective_gap'] = out['cbm'] - np.maximum(out['homo'], out['vbm'])
    out['alignment_regime'] = pd.cut(out['delta_homo_vbm'], [-np.inf, -0.05, 0.15, np.inf], labels=['VBM_above_HOMO', 'near_resonance', 'type_II_separated'])
    return out

def assign_ward_clusters(df, n_clusters=N_WARD_CLUSTERS):
    out = df.copy(); valid = out.dropna(subset=['chain_tail_dist', 'hook_elevation']).copy()
    if len(valid) < n_clusters: raise ValueError('Not enough complete structures to form Ward clusters.')
    labels = fcluster(linkage(StandardScaler().fit_transform(valid[['chain_tail_dist', 'hook_elevation']]), method='ward'), t=n_clusters, criterion='maxclust')
    out['ward_cluster'] = np.nan; out.loc[valid.index, 'ward_cluster'] = labels.astype(int)
    return out

def correlation_table(df, features, targets):
    rows = []
    for target in targets:
        for feature in features:
            sample = df[[feature, target]].dropna()
            rho, p = spearmanr(sample[feature], sample[target]) if len(sample) >= 5 else (np.nan, np.nan)
            rows.append({'feature': feature, 'target': target, 'n': len(sample), 'spearman_rho': rho, 'pvalue': p})
    out = pd.DataFrame(rows); out['pvalue_fdr_bh'] = np.nan
    valid = out.pvalue.notna(); out.loc[valid, 'pvalue_fdr_bh'] = multipletests(out.loc[valid, 'pvalue'], method='fdr_bh')[1]
    return out.sort_values(['target', 'pvalue_fdr_bh'])

def bootstrap_slope(x, y, n_resamples=3000):
    d = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(d) < 6: return {'n': len(d), 'slope': np.nan, 'low': np.nan, 'high': np.nan}
    statistic = lambda xx, yy, axis=-1: linregress(xx, yy).slope
    ci = bootstrap((d.x.to_numpy(), d.y.to_numpy()), statistic, paired=True, n_resamples=n_resamples, random_state=RANDOM_STATE).confidence_interval
    return {'n': len(d), 'slope': linregress(d.x, d.y).slope, 'low': ci.low, 'high': ci.high}

def fit_ols(df, target, features):
    d = df[[target, *features]].dropna()
    if len(d) < len(features) + 10: return None
    return smf.ols(f"{target} ~ {' + '.join(features)} + chain_tail_dist:hook_elevation", data=d).fit(cov_type='HC3')

def fit_nonlinear(df, target, features):
    d = df[[target, *features]].dropna()
    if len(d) < 20: return None, None, None
    model = ExtraTreesRegressor(n_estimators=1000, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1).fit(d[features], d[target])
    perm = permutation_importance(model, d[features], d[target], n_repeats=100, random_state=RANDOM_STATE, scoring='neg_mean_absolute_error', n_jobs=-1)
    importance = pd.DataFrame({'feature': features, 'importance_mae': perm.importances_mean, 'importance_std': perm.importances_std}).sort_values('importance_mae', ascending=False)
    mae = -cross_val_score(model, d[features], d[target], scoring='neg_mean_absolute_error', cv=RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE), n_jobs=-1)
    return model, importance, mae

def plot_results(df, compound, corr, importance):
    fig, ax = plt.subplots(1, 3, figsize=(22, 6))
    sns.scatterplot(data=df, x='chain_tail_dist', y='hook_elevation', hue='ward_cluster', palette='tab10', s=85, ax=ax[0]); ax[0].set_title(f'{compound}: Ward sampling space')
    p1 = ax[1].scatter(df.chain_tail_dist, df.vbm, c=df.hook_elevation, cmap='viridis', s=85, edgecolor='k', linewidth=.25); sns.regplot(data=df, x='chain_tail_dist', y='vbm', scatter=False, color='crimson', ax=ax[1]); ax[1].set_title(f'{compound}: VBM vs distance'); fig.colorbar(p1, ax=ax[1], label='Hook elevation')
    p2 = ax[2].scatter(df.chain_tail_dist, df.hook_elevation, c=df.delta_homo_vbm, cmap='coolwarm', s=85, edgecolor='k', linewidth=.25); ax[2].set_title(f'{compound}: HOMO–VBM map'); fig.colorbar(p2, ax=ax[2], label='HOMO - VBM (eV)')
    fig.tight_layout(); plt.show()
    plt.figure(figsize=(10, 6)); sns.heatmap(corr.pivot(index='feature', columns='target', values='spearman_rho'), cmap='vlag', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f'); plt.title(f'{compound}: Spearman correlations'); plt.tight_layout(); plt.show()
    if importance is not None:
        plt.figure(figsize=(8, 5)); sns.barplot(data=importance, x='importance_mae', y='feature', color='steelblue'); plt.title(f'{compound}: nonlinear importance for VBM'); plt.tight_layout(); plt.show()

def representatives(df, n=5):
    selected = []
    for regime, group in df.groupby('alignment_regime', observed=False):
        if not group.empty:
            item = group.nsmallest(n, 'abs_delta_homo_vbm').copy(); item['selection_reason'] = f'lowest_offset_{regime}'; selected.append(item)
    cols = ['compound', 'structure_id', 'ward_cluster', 'chain_tail_dist', 'hook_elevation', 'vbm', 'homo', 'cbm', 'delta_homo_vbm', 'alignment_regime', 'selection_reason']
    return pd.concat(selected, ignore_index=True)[cols] if selected else pd.DataFrame(columns=cols)

def run_compound(compound, paths):
    satool, dft = load_satool(paths['satool']), load_dft(paths['dft'])
    print('\nParsed DFT records:'); display(dft.head())
    df = satool.merge(dft, on='structure_id', how='inner', validate='one_to_one').copy()
    print(f'{compound}: SATool={len(satool)}, DFT={len(dft)}, merged={len(df)}')
    if len(df) < .8 * len(dft): raise ValueError('Too few DFT rows matched. Verify SATOOL_INDEX_BASE (0 vs 1) and parsed IDs before analysing.')
    df['compound'] = compound; df = assign_ward_clusters(add_electronic_metrics(df))
    features = [feature for feature in SELECTION_RULES if feature in df.columns]
    targets = ['vbm', 'homo', 'cbm', 'delta_homo_vbm', 'abs_delta_homo_vbm', 'inorganic_gap', 'effective_gap']
    corr = correlation_table(df, features, targets); display(df[targets].describe().T); display(corr.head(20))
    slope = bootstrap_slope(df.chain_tail_dist, df.vbm); print(f"VBM slope = {slope['slope']:.4f}; 95% CI [{slope['low']:.4f}, {slope['high']:.4f}]; n={slope['n']}")
    ols_vbm, ols_offset = fit_ols(df, 'vbm', features), fit_ols(df, 'delta_homo_vbm', features)
    if ols_vbm is not None: print(ols_vbm.summary())
    if ols_offset is not None: print(ols_offset.summary())
    _, importance, mae = fit_nonlinear(df, 'vbm', features)
    if mae is not None: print(f'ExtraTrees CV MAE = {mae.mean():.3f} ± {mae.std():.3f} eV'); display(importance)
    plot_results(df, compound, corr, importance)
    reps = representatives(df); display(reps)
    prefix = compound.replace('-', '_'); df.to_csv(OUTPUT_DIR / f'{prefix}_merged_structural_dft.csv', index=False); corr.to_csv(OUTPUT_DIR / f'{prefix}_correlations.csv', index=False); reps.to_csv(OUTPUT_DIR / f'{prefix}_representative_structures.csv', index=False)
    return {'data': df, 'correlations': corr, 'representatives': reps, 'ols_vbm': ols_vbm, 'ols_offset': ols_offset}


## Inspect raw DFT input
Expected parsed IDs include `c3_0025_C0 → 25`, `c3_0155_C2 → 155`, and `c3_1002_C3 → 1002`.


In [ ]:
for compound, paths in COMPOUND_PATHS.items():
    print(f'\n--- {compound}: {paths["dft"]} ---')
    for line in noncomment_lines(paths['dft'])[:3]: print(line)
    display(parse_headerless_dft(paths['dft']).head())


## `cz-c_3` analysis

In [ ]:
results_c3 = run_compound('cz-c_3', COMPOUND_PATHS['cz-c_3'])
df_c3 = results_c3['data']


## `cz-c_4` analysis

In [ ]:
results_c4 = run_compound('cz-c_4', COMPOUND_PATHS['cz-c_4'])
df_c4 = results_c4['data']


## `cz-c_5` analysis

In [ ]:
results_c5 = run_compound('cz-c_5', COMPOUND_PATHS['cz-c_5'])
df_c5 = results_c5['data']


## Pooled analysis

In [ ]:
all_compounds = pd.concat([df_c3, df_c4, df_c5], ignore_index=True)
features = ['chain_tail_dist', 'hook_elevation', 'shared_mxm_angle', 'trans_angle', 'tilt_core', 'tilt_chain']
pooled = all_compounds.dropna(subset=['vbm', 'delta_homo_vbm', 'compound', *features]).copy()
vbm_model = smf.ols('vbm ~ chain_tail_dist * C(compound) + hook_elevation + shared_mxm_angle + trans_angle + tilt_core + tilt_chain', data=pooled).fit(cov_type='HC3')
offset_model = smf.ols('delta_homo_vbm ~ chain_tail_dist * C(compound) + hook_elevation + shared_mxm_angle + trans_angle + tilt_core + tilt_chain', data=pooled).fit(cov_type='HC3')
print(vbm_model.summary()); print(offset_model.summary())
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=all_compounds, x='chain_tail_dist', y='vbm', hue='compound', style='compound', s=90, ax=ax[0])
for _, group in all_compounds.groupby('compound'): sns.regplot(data=group, x='chain_tail_dist', y='vbm', scatter=False, ax=ax[0])
sns.scatterplot(data=all_compounds, x='chain_tail_dist', y='delta_homo_vbm', hue='compound', style='alignment_regime', s=90, ax=ax[1])
ax[1].axhline(0, color='black', ls='--'); ax[1].axhspan(-.05, .15, color='gold', alpha=.18); fig.tight_layout(); plt.show()
all_compounds.to_csv(OUTPUT_DIR / 'all_compounds_merged_structural_dft.csv', index=False)
(OUTPUT_DIR / 'pooled_vbm_model.txt').write_text(vbm_model.summary().as_text())
(OUTPUT_DIR / 'pooled_homo_vbm_offset_model.txt').write_text(offset_model.summary().as_text())
all_representatives = pd.concat([results_c3['representatives'], results_c4['representatives'], results_c5['representatives']], ignore_index=True)
all_representatives.to_csv(OUTPUT_DIR / 'representative_structures_for_HSE_SOC_PDOS.csv', index=False)
display(all_representatives)
